# Qwen3.5-0.8B — Delta-CeNN + CeNN V5.3: all 24 decoder layers

**Research experiment: 18 gated-delta-to-Delta-CeNN substitutions + six full-attention-to-V5.3 substitutions**. Progressive layerwise calibration, paired GPU latency, quality-gated acceptance and automatic results ZIP download. **No upload, no Google Drive.**

The supplied six-full-attention experiment did not pass fidelity or speed gates. This run tests a *new* gated-delta CeNN operator as a separate control. Every layer is checked on the same validation inputs before adding subsequent substitutions. **FORCE_ALL_24** is explicitly diagnostic: when enabled, failed candidates still install and a final failure must not be reported as model success.

Preliminary reference code is eager fp32 PyTorch, **not a fused kernel**, and has no Qwen hybrid-cache support. Text-only, unpadded, no `generate()`.

## 0 · Set NVIDIA GPU, install pinned dependencies and verify both adapter families

The notebook clones `main`, runs CPU contract tests for native/Qwen-gated delta interfaces, and records the pinned model and repository revision. Results are automatically saved as JSONL, CSV, JSON and per-layer PyTorch checkpoints.

In [ ]:
%pip -q install "transformers==5.17.0" "huggingface_hub>=0.36,<2" "datasets>=3,<5" "accelerate>=1" "safetensors" "pandas>=2" "matplotlib>=3.7" "pytest>=8"


In [ ]:
import os,sys,subprocess,gc,json,hashlib,math,random,time,csv,zipfile,traceback
from pathlib import Path
from contextlib import contextmanager,nullcontext
import numpy as np,pandas as pd,torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
REPO=Path('/content/TinyCeNN-LM')
if not (REPO/'.git').is_dir():
    subprocess.run(['git','clone','--quiet','--depth','1','--branch','main',
        'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','--quiet','origin','main'],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
sys.path.insert(0,str(REPO/'src'))
from tinycenn_lm.cenn_v53_multilayer import (FULL_ATTENTION_LAYERS,FullAttentionGraftManager,
    validate_full_attention_targets,check_quality,experiment_fingerprint,append_jsonl)
from tinycenn_lm.delta_cenn_qwen35 import Qwen35DeltaCeNNLinear
from tinycenn_lm.cenn_deltacell_v3 import CeNNDeltaCellV3
from tinycenn_lm.cenn_head_rescue_v5 import Qwen35HeadRescueV5
from tinycenn_lm.cenn_head_rescue_v51 import CeNNQueryRescueV51,warmstart_v5_delta_core
from tinycenn_lm.cenn_head_rescue_v52 import CeNNHeadRescueV52,warmstart_v51_to_v52
from tinycenn_lm.cenn_head_rescue_v53 import CeNNHeadRescueV53,warmstart_v52_to_v53
from transformers import AutoConfig,AutoTokenizer,Qwen3_5ForCausalLM
from huggingface_hub import model_info
subprocess.run([sys.executable,'-m','pytest','-q',
    str(REPO/'tests/test_delta_cenn_qwen35.py'),
    str(REPO/'tests/test_cenn_head_rescue_v53.py'),
    str(REPO/'tests/test_qwen35_v53_all24_contract.py')],check=True)
assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → NVIDIA GPU'
DEVICE=torch.device('cuda')
DTYPE=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
GIT_SHA=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
SOURCE_FILES=[REPO/'src/tinycenn_lm'/f for f in ('delta_cenn_qwen35.py',
    'cenn_v53_multilayer.py','cenn_deltacell_v3.py','cenn_head_rescue_v5.py',
    'cenn_head_rescue_v51.py','cenn_head_rescue_v52.py','cenn_head_rescue_v53.py')]
SOURCE_SHA256=hashlib.sha256(b''.join(p.read_bytes() for p in SOURCE_FILES)).hexdigest()
print('Git:',GIT_SHA,'model dtype:',DTYPE,'device:',torch.cuda.get_device_name(0))


## 1 · Reproducible, all-24-layer experiment settings

`PROFILE='quick'` performs contract and feasibility screening, not publication-grade benchmarking. `FORCE_ALL_24=True` installs every trained candidate **even if rejected** to measure cumulative degradation; turn it off to accept only stages meeting the accuracy gates. `DELTA_FEATURE_DIM=None` preserves Qwen's complete key representation; smaller dimensions study compression at increased fidelity risk.

In [ ]:
# @title 24-layer Delta-CeNN / CeNN V5.3 research controls
PROFILE='quick' # @param ['quick','research']
FORCE_ALL_24=True # @param {type:'boolean'}
REPLACEMENT_MODE='one_group' # @param ['one_group','both_groups']
AUTO_DOWNLOAD_RESULTS=True # @param {type:'boolean'}
MODEL_ID='Qwen/Qwen3.5-0.8B'
SEQ_LEN=64 if PROFILE=='quick' else 128
TRAIN_BLOCKS=16 if PROFILE=='quick' else 64
DEV_BLOCKS=8 if PROFILE=='quick' else 32
AUDIT_BLOCKS=8 if PROFILE=='quick' else 32
AUDIT_START_TOKEN=1_850_000
BATCH_SIZE=1 if PROFILE=='quick' else 2
DELTA_STEPS=8 if PROFILE=='quick' else 60
BASELINE_STEPS=24 if PROFILE=='quick' else 180
V53_STEPS=32 if PROFILE=='quick' else 320
DELTA_LR=2e-4
BASELINE_LR=2e-3
V53_LR=6e-4
DELTA_FEATURE_DIM=None  # None = native full-rank key features; compressed < key dimension is experimental
DELTA_CELL_INTERVAL=16
FEATURE_DIM=64
LOCAL_WINDOW=64 if PROFILE=='quick' else 128
CHUNK_SIZE=32
RANK=16
SEED=5318
LOG_EVERY=4 if PROFILE=='quick' else 15
MIN_COSINE=.95;MIN_AGREEMENT=.98;MAX_DELTA_NLL=.10;MIN_SPEEDUP=1.05
OUT=Path('/content/qwen35_delta_cenn_all24_results')
OUT.mkdir(parents=True,exist_ok=True)
(OUT/'checkpoints').mkdir(exist_ok=True)
TRAIN_EVENTS=OUT/'training_steps.jsonl'
TEXT_LOG=OUT/'training.log'
STAGE_CSV=OUT/'cumulative_layer_results.csv'
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32=True
print('Full-attention mode',REPLACEMENT_MODE,'force all 24',FORCE_ALL_24)


## 2 · Load and inspect the original Qwen3.5 hybrid backbone

Verify **six** full-attention layers (SDPA, special V5.3 wrapper) and **18** gated-delta layers (`linear_attn`, Delta-CeNN wrapper). Pretrained projection and MLP weights remain frozen. Validate wrapper numerical parity before substituting any layer.

In [ ]:
REVISION=model_info(MODEL_ID).sha
cfg=AutoConfig.from_pretrained(MODEL_ID,revision=REVISION).get_text_config(decoder=True)
assert cfg.model_type=='qwen3_5_text' and cfg.hidden_size==1024 and cfg.num_hidden_layers==24
FULL_TARGETS=tuple(i for i,t in enumerate(cfg.layer_types) if t=='full_attention')
DELTA_TARGETS=tuple(i for i,t in enumerate(cfg.layer_types) if t=='linear_attention')
assert FULL_TARGETS==FULL_ATTENTION_LAYERS and len(DELTA_TARGETS)==18
TARGET_LAYERS=tuple(range(cfg.num_hidden_layers))
assert sorted(FULL_TARGETS+DELTA_TARGETS)==list(TARGET_LAYERS)
TOKENIZER=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,use_fast=True)
model=Qwen3_5ForCausalLM.from_pretrained(MODEL_ID,revision=REVISION,dtype=DTYPE,
    attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
model.requires_grad_(False);model.config.use_cache=False
LAYERS=model.model.layers
MANAGER=FullAttentionGraftManager(LAYERS,FULL_TARGETS)
DELTA_NATIVE={i:LAYERS[i].linear_attn for i in DELTA_TARGETS}
INSTALLED_DELTA={}
HD=int(MANAGER.native[11].head_dim)
QHEADS=int(MANAGER.native[11].q_proj.out_features//(2*HD))
KVHEADS=int(MANAGER.native[11].k_proj.out_features//HD)
assert (QHEADS,KVHEADS,HD)==(8,2,256)
EXACT_GROUPS=(0,) if REPLACEMENT_MODE=='one_group' else ()
APPROX_KV=KVHEADS-len(EXACT_GROUPS)
APPROX_Q=APPROX_KV*(QHEADS//KVHEADS)
@contextmanager
def native_everywhere():
    restored={i:LAYERS[i].linear_attn for i in DELTA_TARGETS}
    with MANAGER.native_context():
        try:
            for i,original in DELTA_NATIVE.items():LAYERS[i].linear_attn=original
            yield
        finally:
            for i,old in restored.items():LAYERS[i].linear_attn=old
@contextmanager
def delta_candidate(idx,adapter):
    assert idx in DELTA_NATIVE and adapter.original is DELTA_NATIVE[idx]
    assert LAYERS[idx].linear_attn is DELTA_NATIVE[idx]
    LAYERS[idx].linear_attn=adapter
    try:yield
    finally:LAYERS[idx].linear_attn=DELTA_NATIVE[idx]
def install_delta(idx,adapter):
    assert idx in DELTA_NATIVE and adapter.original is DELTA_NATIVE[idx]
    assert LAYERS[idx].linear_attn is DELTA_NATIVE[idx]
    LAYERS[idx].linear_attn=adapter
    INSTALLED_DELTA[idx]=adapter
def installed_indices():
    return sorted(set(MANAGER.installed_layers)|set(INSTALLED_DELTA))
# Numerical parity controls on a text-only prefix, before training.
probe=TOKENIZER('Delta CeNN compares causal learned memory with original Qwen.',
    add_special_tokens=False,return_tensors='pt').input_ids.to(DEVICE)
with torch.inference_mode(),torch.autocast('cuda',dtype=DTYPE):
    original_logits=model(input_ids=probe,use_cache=False,logits_to_keep=0).logits.float()
    oracle=Qwen35HeadRescueV5(MANAGER.native[FULL_TARGETS[0]],None,exact_groups=tuple(range(KVHEADS))).eval()
    with MANAGER.candidate(FULL_TARGETS[0],oracle):
        exact_logits=model(input_ids=probe,use_cache=False,logits_to_keep=0).logits.float()
full_parity=float((original_logits-exact_logits).abs().max())
assert full_parity<.05 and bool((original_logits.argmax(-1)==exact_logits.argmax(-1)).all())
linear_idx=DELTA_TARGETS[0]
delta_oracle=Qwen35DeltaCeNNLinear(DELTA_NATIVE[linear_idx],feature_dim=None,cells=False).to(DEVICE).eval()
with torch.inference_mode(),torch.autocast('cuda',dtype=DTYPE):
    with delta_candidate(linear_idx,delta_oracle):
        delta_logits=model(input_ids=probe,use_cache=False,logits_to_keep=0).logits.float()
delta_parity=float((original_logits-delta_logits).abs().max())
# We allow kernel/numerical-order differences, but fail fast on silent interface drift.
assert delta_parity<.15, f'Delta-CeNN parity failed on this backend: {delta_parity}'
print('Pinned model',REVISION,'full layers',FULL_TARGETS,'delta layers',DELTA_TARGETS)
print('Exact full-attention parity:',full_parity,'full-rank Delta-CeNN parity:',delta_parity)
del oracle,delta_oracle,probe,original_logits,exact_logits,delta_logits;torch.cuda.empty_cache()


## 3 · Data provenance and frozen native-teacher baseline

Train on WikiText-2 train prefix, rank and gate on validation, then audit once on a predeclared disjoint portion of the train partition (not an independent blinded benchmark). Prior test-split results are not reused. All 24 stages share a single immutable training/validation data selection.

In [ ]:
from datasets import load_dataset
def token_windows(split,count,length,start_token=0):
    ds=load_dataset('Salesforce/wikitext','wikitext-2-raw-v1',split=split)
    needed=start_token+count*length
    ids=[]
    for row in ds:
        sentence=row['text'].strip()
        if not sentence:continue
        ids.extend(TOKENIZER.encode(sentence,add_special_tokens=False))
        ids.append(TOKENIZER.eos_token_id)
        if len(ids)>=needed:break
    if len(ids)<needed:raise RuntimeError(f'{split} has {len(ids)} tokens; need {needed}')
    return torch.tensor(ids[start_token:needed],dtype=torch.long).reshape(count,length)
TRAIN=token_windows('train',TRAIN_BLOCKS,SEQ_LEN)
DEV=token_windows('validation',DEV_BLOCKS,SEQ_LEN)
DATA_HASHES={name:hashlib.sha256(data.numpy().tobytes()).hexdigest()
    for name,data in (('train',TRAIN),('validation',DEV))}
assert TRAIN_BLOCKS*SEQ_LEN<AUDIT_START_TOKEN
CONFIG=dict(model=MODEL_ID,revision=REVISION,repo_sha=GIT_SHA,source_sha256=SOURCE_SHA256,
    profile=PROFILE,force_all24=FORCE_ALL_24,replacement_mode=REPLACEMENT_MODE,
    all_targets=TARGET_LAYERS,full_targets=FULL_TARGETS,delta_targets=DELTA_TARGETS,
    seq_len=SEQ_LEN,train_blocks=TRAIN_BLOCKS,dev_blocks=DEV_BLOCKS,
    audit_blocks=AUDIT_BLOCKS,audit_offset=AUDIT_START_TOKEN,
    delta_steps=DELTA_STEPS,baseline_steps=BASELINE_STEPS,v53_steps=V53_STEPS,
    delta_feature_dim=DELTA_FEATURE_DIM,delta_cell_interval=DELTA_CELL_INTERVAL,
    batch_size=BATCH_SIZE,local_window=LOCAL_WINDOW,seed=SEED,data_hashes=DATA_HASHES,
    torch_version=torch.__version__,gpu=torch.cuda.get_device_name(0))
RUN_FINGERPRINT=experiment_fingerprint(CONFIG)
(OUT/'config.json').write_text(json.dumps(CONFIG,indent=2,allow_nan=False))
print('Train',TRAIN.shape,'Validation',DEV.shape,'fingerprint',RUN_FINGERPRINT[:16])


## 4 · Sequential teacher capture, full-model metrics, training logs and paired latency

Each target layer is calibrated using the **actual upstream prefix already installed**. Distinct capture interfaces preserve full-attention RoPE and gated-delta tensor signatures. Every training step goes to `training_steps.jsonl`, and each stage is logged to `training.log`. Native timing temporarily restores **both** full-attention and linear-attention modules.

In [ ]:
def log_line(message):
    line=f'[{time.strftime("%Y-%m-%dT%H:%M:%S")}] {message}'
    print(line,flush=True)
    with TEXT_LOG.open('a',encoding='utf-8') as f:f.write(line+'\n')
@torch.inference_mode()
def inference(blocks,keep=1):
    with torch.autocast('cuda',dtype=DTYPE):
        return model(input_ids=blocks.to(DEVICE),use_cache=False,logits_to_keep=keep)
@torch.inference_mode()
def capture_target(idx,split,blocks):
    full=idx in FULL_TARGETS
    native=MANAGER.native[idx] if full else DELTA_NATIVE[idx]
    assert (LAYERS[idx].self_attn if full else LAYERS[idx].linear_attn) is native
    stored={k:[] for k in (('x','y','cos','sin') if full else ('x','y'))}
    def hook(mod,args,kwargs,out):
        x=kwargs.get('hidden_states')
        if x is None and args:x=args[0]
        y=out[0] if isinstance(out,tuple) else out
        stored['x'].append(x.detach().to('cpu',dtype=torch.float16))
        stored['y'].append(y.detach().to('cpu',dtype=torch.float16))
        if full:
            cos,sin=kwargs['position_embeddings']
            stored['cos'].append(cos.detach().cpu().float())
            stored['sin'].append(sin.detach().cpu().float())
    handle=native.register_forward_hook(hook,with_kwargs=True)
    try:
        for start in range(0,len(blocks),BATCH_SIZE):inference(blocks[start:start+BATCH_SIZE])
    finally:handle.remove()
    answer={key:torch.cat(values,0).contiguous() for key,values in stored.items()}
    assert answer['x'].shape[:2]==blocks.shape
    return answer
def captured_batch(data,indices):
    x=data['x'][indices].to(DEVICE,dtype=DTYPE)
    y=data['y'][indices].to(DEVICE,dtype=DTYPE)
    rope=(data['cos'][indices].to(DEVICE),data['sin'][indices].to(DEVICE)) if 'cos' in data else None
    return x,y,rope
def forward_adapter(adapter,x,rope):
    if isinstance(adapter,Qwen35DeltaCeNNLinear):return adapter(x)
    output,_=adapter(x,position_embeddings=rope)
    return output
def mixer_loss(pred,teacher):
    p=pred.float();t=teacher.float()
    nmse=(p-t).square().mean()/t.square().mean().clamp_min(1e-8)
    return nmse+.2*(1-F.cosine_similarity(p,t,dim=-1).mean())
@torch.inference_mode()
def local_validation(adapter,data):
    adapter.eval();err=energy=cosine_sum=0.;n=0
    for lo in range(0,len(data['x']),BATCH_SIZE):
        indices=torch.arange(lo,min(lo+BATCH_SIZE,len(data['x'])))
        x,y,rope=captured_batch(data,indices)
        with torch.autocast('cuda',dtype=DTYPE):pred=forward_adapter(adapter,x,rope)
        p,t=pred.float(),y.float()
        err+=(p-t).square().sum().item();energy+=t.square().sum().item()
        cosine_sum+=F.cosine_similarity(p,t,dim=-1).sum().item()
        n+=p.shape[0]*p.shape[1]
    return dict(nmse=err/max(energy,1e-8),cosine=cosine_sum/max(n,1))
@torch.inference_mode()
def model_metrics(blocks,teacher_top1=None):
    n=0;nll=0.;correct=0;agreement=0;tops=[]
    for lo in range(0,len(blocks),BATCH_SIZE):
        block=blocks[lo:lo+BATCH_SIZE]
        positions=torch.arange(block.shape[1]-1,device=DEVICE)
        logits=inference(block,keep=positions).logits.float()
        gold=block[:,1:].to(DEVICE)
        logp=F.log_softmax(logits,dim=-1)
        nll-=logp.gather(-1,gold.unsqueeze(-1)).sum().item()
        pred=logits.argmax(-1)
        correct+=(pred==gold).sum().item();n+=gold.numel()
        if teacher_top1 is None:tops.append(pred.cpu().to(torch.int32))
        else:
            reference=teacher_top1[lo//BATCH_SIZE].to(DEVICE)
            assert reference.shape==pred.shape
            agreement+=(reference==pred).sum().item()
    out=dict(nll=nll/n,accuracy=correct/n,decisions=n)
    if teacher_top1 is not None:out['teacher_agreement']=agreement/n
    return out,tops
@torch.inference_mode()
def paired_prefill(blocks,warmups=2,repeats=6):
    sample=blocks[:min(BATCH_SIZE,len(blocks))].to(DEVICE)
    native_times=[];student_times=[]
    def one(native):
        with native_everywhere() if native else nullcontext():
            begin=torch.cuda.Event(enable_timing=True);end=torch.cuda.Event(enable_timing=True)
            begin.record();out=inference(sample,keep=1).logits
            end.record();torch.cuda.synchronize()
            assert bool(torch.isfinite(out).all())
            return begin.elapsed_time(end)
    for i in range(warmups+repeats):
        if i%2:student=one(False);native=one(True)
        else:native=one(True);student=one(False)
        if i>=warmups:native_times.append(native);student_times.append(student)
    return dict(native_ms=float(np.median(native_times)),
        student_ms=float(np.median(student_times)),
        speedup=float(np.median(native_times)/np.median(student_times)),
        native_iqr_ms=float(np.percentile(native_times,75)-np.percentile(native_times,25)),
        student_iqr_ms=float(np.percentile(student_times,75)-np.percentile(student_times,25)),
        repeats=repeats)


## 5 · Two architecture families, one progressive experiment

- **Linear-attention layers (18)**: `Qwen35DeltaCeNNLinear` reuses Qwen convolution/projections and substitutes its gated-delta recurrence with a causal FP32 Delta-CeNN cellular matrix. Default full-rank features preserve the native key dimension; a learned, bounded neighbor-cell update is trainable.
- **Full-attention layers (6)**: existing V5.3 Head Rescue, a trained gated-delta feature core with local softmax window and cross-calibration, optionally preserving one exact KV group.

Train each layer once in increasing layer order, test the cumulative prefix, record failures and always keep source checkpoints.

In [ ]:
def make_base(idx):
    core=CeNNDeltaCellV3(APPROX_Q,APPROX_KV,HD,
        feature_dim=FEATURE_DIM,chunk_size=CHUNK_SIZE,mode='none',
        local_window=LOCAL_WINDOW).to(DEVICE)
    return Qwen35HeadRescueV5(MANAGER.native[idx],core,exact_groups=EXACT_GROUPS).eval()
def make_v53(idx,base_state):
    common=dict(num_heads=APPROX_Q,num_kv_heads=APPROX_KV,head_dim=HD,
        feature_dim=FEATURE_DIM,chunk_size=CHUNK_SIZE,local_window=LOCAL_WINDOW,
        correction_rank=RANK,query_router=True,query_correction=True)
    v51=CeNNQueryRescueV51(**common);warmstart_v5_delta_core(v51,base_state)
    v52=CeNNHeadRescueV52(**common,post_rank=RANK,post_correction=True)
    warmstart_v51_to_v52(v52,v51.state_dict())
    v53=CeNNHeadRescueV53(**common,post_rank=RANK,post_correction=True,
        cross_rank=RANK,cross_correction=True,output_calibration=True)
    warmstart_v52_to_v53(v53,v52.state_dict())
    return Qwen35HeadRescueV5(MANAGER.native[idx],v53.to(DEVICE),exact_groups=EXACT_GROUPS).eval()
def make_delta(idx):
    return Qwen35DeltaCeNNLinear(DELTA_NATIVE[idx],feature_dim=DELTA_FEATURE_DIM,
        cells=True,cell_interval=DELTA_CELL_INTERVAL).to(DEVICE).eval()
def fit_local(idx,stage,adapter,train,dev,steps,lr):
    core=adapter.core
    core.requires_grad_(True)
    optimizer=torch.optim.AdamW(core.parameters(),lr=lr,weight_decay=.005)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=steps,eta_min=lr*.1)
    scaler=torch.amp.GradScaler('cuda',enabled=(DTYPE==torch.float16))
    generator=torch.Generator().manual_seed(SEED+idx*29+(0 if stage=='delta_cenn' else 5000))
    indices=torch.randint(0,len(train['x']),(steps,BATCH_SIZE),generator=generator)
    best=math.inf;best_sd=None;best_local=None
    began=time.perf_counter()
    for step in range(1,steps+1):
        ix=indices[step-1];x,y,rope=captured_batch(train,ix)
        core.train();optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda',dtype=DTYPE):
            pred=forward_adapter(adapter,x,rope)
            loss=mixer_loss(pred,y)
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError(f'L{idx}/{stage} step {step}: non-finite loss')
        scaler.scale(loss).backward();scaler.unscale_(optimizer)
        grad=torch.nn.utils.clip_grad_norm_(core.parameters(),1.)
        scaler.step(optimizer);scaler.update();scheduler.step()
        event=dict(layer=idx,family='linear_attention' if idx in DELTA_TARGETS else 'full_attention',
            stage=stage,step=step,total_steps=steps,loss=float(loss.detach()),
            grad_norm=float(grad),lr=float(optimizer.param_groups[0]['lr']),
            elapsed_seconds=float(time.perf_counter()-began))
        if step==1 or step==steps or step%LOG_EVERY==0:
            stats=local_validation(adapter,dev)
            score=stats['nmse']+.2*(1-stats['cosine'])
            if score<best:
                best=score;best_local=stats
                best_sd={k:v.detach().cpu().clone() for k,v in core.state_dict().items()}
            event.update(val_cosine=stats['cosine'],val_nmse=stats['nmse'])
            log_line(f'L{idx} {stage} {step}/{steps} loss={event["loss"]:.6f} '
                f'cos={stats["cosine"]:.5f} nmse={stats["nmse"]:.5f}')
        append_jsonl(TRAIN_EVENTS,event)
    assert best_sd is not None
    core.load_state_dict(best_sd,strict=True);core.requires_grad_(False);adapter.eval()
    return best_sd,best_local
@torch.inference_mode()
def warmstart_parity(base,updated,dev):
    x,_,rope=captured_batch(dev,torch.tensor([0]))
    with torch.autocast('cuda',dtype=DTYPE):
        a=forward_adapter(base,x,rope);b=forward_adapter(updated,x,rope)
    diff=float((a-b).abs().max())
    assert diff<.01,diff
    return diff


## 6 · Run 24 sequential replacements and auto-download all training statistics

The results archive is exported inside a `finally` block, even on interrupted runs that raise a Python exception. Google Colab may require your browser to allow its automatic download prompt. The native benchmark restores **all 24 original layers**, not merely the six attention modules.

In [ ]:
REPORT=[];RUN_OK=False;RUN_ERROR=None;RUN_START=time.time()
NATIVE_DEV=None;FINAL=None;ARCHIVE=None
try:
    log_line('START Delta-CeNN + CeNN V5.3 / all 24 '+RUN_FINGERPRINT[:16])
    with native_everywhere():NATIVE_DEV,NATIVE_TOP1=model_metrics(DEV)
    log_line('NATIVE validation '+json.dumps(NATIVE_DEV))
    for layer in TARGET_LAYERS:
        family='full_attention' if layer in FULL_TARGETS else 'linear_attention'
        log_line(f'BEGIN layer {layer} {family}; installed prefix {installed_indices()}')
        train_capture=capture_target(layer,'train',TRAIN)
        dev_capture=capture_target(layer,'dev',DEV)
        torch.manual_seed(SEED+layer*13)
        warmstart_diff=0.0
        if family=='linear_attention':
            adapter=make_delta(layer)
            state,local=fit_local(layer,'delta_cenn',adapter,
                train_capture,dev_capture,DELTA_STEPS,DELTA_LR)
            steps=dict(delta_cenn=DELTA_STEPS)
        else:
            baseline=make_base(layer)
            base_state,_=fit_local(layer,'full_delta_seed',baseline,
                train_capture,dev_capture,BASELINE_STEPS,BASELINE_LR)
            torch.save(dict(core_state=base_state,layer=layer,stage='full_delta_seed',
                fingerprint=RUN_FINGERPRINT),OUT/'checkpoints'/f'L{layer:02d}_full_seed.pt')
            adapter=make_v53(layer,base_state)
            warmstart_diff=warmstart_parity(baseline,adapter,dev_capture)
            del baseline
            torch.cuda.empty_cache()
            state,local=fit_local(layer,'full_v53',adapter,
                train_capture,dev_capture,V53_STEPS,V53_LR)
            steps=dict(full_delta_seed=BASELINE_STEPS,full_v53=V53_STEPS)
        checkpoint=OUT/'checkpoints'/f'L{layer:02d}_{family}_core.pt'
        torch.save(dict(core_state=state,layer=layer,family=family,
            core_config=adapter.core.config if hasattr(adapter.core,'config') else dict(
                feature_dim=adapter.core.feature_dim,cell_interval=adapter.core.cell_interval),
            validation=local,fingerprint=RUN_FINGERPRINT,model_revision=REVISION,
            source_git_sha=GIT_SHA,replacement_mode=REPLACEMENT_MODE),checkpoint)
        context=MANAGER.candidate(layer,adapter) if family=='full_attention' else delta_candidate(layer,adapter)
        with context:
            student_dev,_=model_metrics(DEV,teacher_top1=NATIVE_TOP1)
            # Timing within the candidate context includes this layer plus installed prefix.
            timing=paired_prefill(DEV,warmups=2,repeats=4 if PROFILE=='quick' else 12)
        delta_nll=student_dev['nll']-NATIVE_DEV['nll']
        gate=check_quality(local['cosine'],student_dev['teacher_agreement'],delta_nll,
            timing['speedup'],min_cosine=MIN_COSINE,min_agreement=MIN_AGREEMENT,
            max_delta_nll=MAX_DELTA_NLL,min_speedup=MIN_SPEEDUP)
        install=bool(FORCE_ALL_24 or gate.quality_pass)
        if install:
            if family=='full_attention':MANAGER.install(layer,adapter)
            else:install_delta(layer,adapter)
        row=dict(layer=layer,family=family,installed=install,
            forced=bool(install and not gate.quality_pass),
            installed_count=len(installed_indices()),
            local_cosine=local['cosine'],local_nmse=local['nmse'],
            cumulative_dev_nll=student_dev['nll'],dev_delta_nll=delta_nll,
            cumulative_dev_agreement=student_dev['teacher_agreement'],
            cumulative_dev_accuracy=student_dev['accuracy'],
            native_ms=timing['native_ms'],student_ms=timing['student_ms'],
            cumulative_speedup=timing['speedup'],quality_pass=gate.quality_pass,
            deploy_pass=gate.deploy_pass,warmstart_max_abs_diff=warmstart_diff,
            trainable_core_params=sum(p.numel() for p in adapter.core.parameters()),**steps)
        REPORT.append(row);pd.DataFrame(REPORT).to_csv(STAGE_CSV,index=False)
        log_line('LAYER SUMMARY '+json.dumps(row,allow_nan=False))
        del state,train_capture,dev_capture
        gc.collect();torch.cuda.empty_cache()
    AUDIT=token_windows('train',AUDIT_BLOCKS,SEQ_LEN,start_token=AUDIT_START_TOKEN)
    DATA_HASHES['diagnostic_train_tail']=hashlib.sha256(AUDIT.numpy().tobytes()).hexdigest()
    assert len(set(DATA_HASHES.values()))==3
    with native_everywhere():NATIVE_AUDIT,AUDIT_TOP1=model_metrics(AUDIT)
    STUDENT_AUDIT,_=model_metrics(AUDIT,teacher_top1=AUDIT_TOP1)
    FINAL_TIMING=paired_prefill(AUDIT,warmups=4,repeats=16)
    all24=len(installed_indices())==24
    stage_quality=all(row['quality_pass'] for row in REPORT if row['installed'])
    audit_delta=STUDENT_AUDIT['nll']-NATIVE_AUDIT['nll']
    FINAL_PASS=bool(all24 and stage_quality and audit_delta<=MAX_DELTA_NLL
                    and STUDENT_AUDIT['teacher_agreement']>=MIN_AGREEMENT
                    and FINAL_TIMING['speedup']>MIN_SPEEDUP)
    FINAL=dict(architecture='Qwen3.5-0.8B: 18 Delta-CeNN + six V5.3 head rescue',
       model_revision=REVISION,source_git_sha=GIT_SHA,source_sha256=SOURCE_SHA256,
       installed=installed_indices(),full_attention_targets=FULL_TARGETS,
       gated_delta_targets=DELTA_TARGETS,forced_all_24=FORCE_ALL_24,
       all24_installed=all24,all_stage_quality_pass=stage_quality,final_pass=FINAL_PASS,
       native_validation=NATIVE_DEV,stage_results=REPORT,native_audit=NATIVE_AUDIT,
       student_audit=STUDENT_AUDIT,audit_delta_nll=audit_delta,
       paired_audit=FINAL_TIMING,audit_source='disjoint WikiText-2 train tail, not external test',
       data_hashes=DATA_HASHES,run_fingerprint=RUN_FINGERPRINT,
       runtime_seconds=time.time()-RUN_START,
       limitations='Eager non-fused fp32 Delta-CeNN recurrent prototype; text-only prefill; no Qwen hybrid KV cache or generation; forced mode is diagnostic.')
    (OUT/'final_audit.json').write_text(json.dumps(FINAL,indent=2,allow_nan=False))
    log_line('FINAL AUDIT '+json.dumps({k:FINAL[k] for k in (
        'all24_installed','all_stage_quality_pass','final_pass','audit_delta_nll','paired_audit')},allow_nan=False))
    if not FINAL_PASS:log_line('REJECT for deployment: research 24-layer artifacts saved, native Qwen remains the reference.')
    RUN_OK=True
except BaseException as exc:
    RUN_ERROR=repr(exc)
    log_line('EXPERIMENT FAILED (preserving completed layers): '+RUN_ERROR)
    with (OUT/'error_traceback.txt').open('w') as f:traceback.print_exc(file=f)
    raise
finally:
    pd.DataFrame(REPORT).to_csv(STAGE_CSV,index=False)
    status=dict(completed=RUN_OK,error=RUN_ERROR,installed=installed_indices(),
        run_fingerprint=RUN_FINGERPRINT,runtime_seconds=time.time()-RUN_START)
    (OUT/'run_status.json').write_text(json.dumps(status,indent=2,allow_nan=False))
    if REPORT:
        try:
            f=pd.DataFrame(REPORT);fig,ax=plt.subplots(figsize=(11,4))
            ax.plot(f.layer,f.cumulative_dev_agreement,marker='.',label='Teacher agreement')
            ax.axhline(MIN_AGREEMENT,ls='--',color='gray')
            ax.set(xlabel='Decoder layer replaced',ylabel='Native top-1 agreement',
                title='Cumulative 24-layer Delta-CeNN + V5.3 experiment')
            ax.legend();fig.tight_layout();fig.savefig(OUT/'cumulative_agreement.png',dpi=160)
            plt.close(fig)
        except Exception as plot_error:log_line('Plot error: '+repr(plot_error))
    manifest=[]
    for path in sorted(OUT.rglob('*')):
        if path.is_file() and path.name!='artifact_manifest.json':
            manifest.append(dict(path=str(path.relative_to(OUT)),bytes=path.stat().st_size,
                sha256=hashlib.sha256(path.read_bytes()).hexdigest()))
    (OUT/'artifact_manifest.json').write_text(json.dumps(manifest,indent=2))
    archive_path=OUT.parent/(OUT.name+'.zip')
    with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as archive:
        for path in sorted(OUT.rglob('*')):
            if path.is_file():archive.write(path,arcname=str(path.relative_to(OUT)))
    ARCHIVE=str(archive_path)
    print('RESULT ARCHIVE',ARCHIVE,'size',archive_path.stat().st_size,flush=True)
    print('Contains checkpoints, training_steps.jsonl, training.log, cumulative_layer_results.csv, final_audit.json (if complete), and checksums.')
    if AUTO_DOWNLOAD_RESULTS:
        try:
            from google.colab import files
            files.download(ARCHIVE)
        except ImportError:
            from IPython.display import display,FileLink
            display(FileLink(ARCHIVE))
        except Exception as e:
            print('Browser blocked automatic download. Retrieve from Colab Files:',ARCHIVE,'reason',str(e))


### Interpretation and reproducibility

- `all24_installed=True` means every substitution was applied; it does **not** establish fidelity or a speedup.
- `final_pass` requires **each installed stage** to pass its accuracy gate, plus final audit agreement ≥0.98, ΔNLL ≤0.10 and actual end-to-end speedup >1.05×.
- `linear_attention` experiments *replace the native gated-delta recurrence*. The full-rank, no-cell control is intended as a numerical parity test; trained cellular coupling is separately measured.
- The new Delta-CeNN is a research Python implementation. 24-layer execution may be substantially slower than optimized kernels. No fused CUDA or cached generation is implemented.
- Archive download is triggered automatically on success or catchable failure. A browser may still block a download; recover the ZIP from the Colab Files sidebar.
- Results include complete training step events and cumulative metrics; research claims require more seeds and genuinely independent evaluation.
